## Training notebook for ml algorithms
- Comparing all models after hyperparameter tuning all the models

    - Importing libraries

In [5]:
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    - Importing dataset

In [6]:
data = pd.read_csv("../../data/traffic_data_cleaned_&_engineered.csv")
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,traffic_volume,is_holiday,day_of_week,hour_sin,hour_cos,month_sin,month_cos
0,288.28,0.0,0.0,40,Clouds,5545,0,1,7.071068e-01,-0.707107,-0.866025,0.5
1,289.36,0.0,0.0,75,Clouds,4516,0,1,5.000000e-01,-0.866025,-0.866025,0.5
2,289.58,0.0,0.0,90,Clouds,4767,0,1,2.588190e-01,-0.965926,-0.866025,0.5
3,290.13,0.0,0.0,90,Clouds,5026,0,1,1.224647e-16,-1.000000,-0.866025,0.5
4,291.14,0.0,0.0,75,Clouds,4918,0,1,-2.588190e-01,-0.965926,-0.866025,0.5


### Creating pipeline for training

    - Dividing categorical and numerical features

In [7]:
categorical_features = ['weather_main']

numerical_features = ['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'is_holiday', 'day_of_week', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']

    - Defining column transformer

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop' # Drops any columns not explicitly defined above (like the target)
)

    - Creating pipelines for each algorithm

    - Elastic Linear regression

In [9]:
elastic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', ElasticNet(random_state=42))
])

    - Random Forest

In [10]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1)) # n_jobs=-1 uses all CPU cores
])

    - XGBoost

In [11]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(tree_method='hist', device='cuda', random_state=42)) # Using GPU for Colab
])

    - Dividing Independent and dependent variables

In [12]:
X = data.drop('traffic_volume', axis=1)
y = data['traffic_volume']

    - Dividing train test split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    - Making a function to automatically train and evaluate all the models at once

In [14]:
def train_and_evaluate(pipelines_dict, X_train, y_train, X_test, y_test, optimize_for='rmse'):
    """
    Trains multiple sklearn pipelines, evaluates them, and returns the best one.

    Parameters:
    pipelines_dict (dict): Dictionary of { 'Model Name': pipeline_object }
    X_train, y_train, X_test, y_test: Training and testing data
    optimize_for (str): The metric to use to select the best model ('rmse', 'mae', or 'r2')

    Returns:
    tuple: (Best Pipeline Object, DataFrame of all results)
    """
    print(f"Training and evaluating {len(pipelines_dict)} models...\n")

    # Dictionary to store results for the dataframe later
    results_data = []

    # Track the best model
    best_model_name = None

    # Set initial best score (infinity for errors, negative infinity for R2)
    if optimize_for in ['rmse', 'mae']:
        best_score = float('inf')
    elif optimize_for == 'r2':
        best_score = float('-inf')
    else:
        raise ValueError("optimize_for must be 'rmse', 'mae', or 'r2'")

    # Loop through each pipeline
    for name, pipeline in pipelines_dict.items():
        # 1. Train the model
        pipeline.fit(X_train, y_train)

        # 2. Make predictions
        predictions = pipeline.predict(X_test)

        # 3. Calculate Metrics
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)

        # Store results
        results_data.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

        # 4. Check if this is the new "best" model
        if optimize_for == 'rmse' and rmse < best_score:
            best_score = rmse
            best_model_name = name
        elif optimize_for == 'mae' and mae < best_score:
            best_score = mae
            best_model_name = name
        elif optimize_for == 'r2' and r2 > best_score:
            best_score = r2
            best_model_name = name

    # Convert results to a clean Pandas DataFrame for nice formatting
    results_df = pd.DataFrame(results_data).set_index('Model')

    # Print the final report
    print("-" * 50)
    print("📋 EVALUATION RESULTS")
    print("-" * 50)
    print(results_df.round(4)) # Round to 4 decimal places for readability
    print("-" * 50)
    print(f"🏆 BEST MODEL ({optimize_for.upper()}): {best_model_name} with score: {best_score:.4f}")

    # Return the actual trained pipeline of the winning model so you can use it
    return pipelines_dict[best_model_name], results_df

    - Training and comparing all the models

In [15]:
my_pipelines = {
    'ElasticNet': elastic_pipeline,
    'Random Forest': rf_pipeline,
    'XGBoost': xgb_pipeline
}
# Let's say we want the model that minimizes the Root Mean Squared Error (RMSE)
best_pipeline, all_results = train_and_evaluate(
    pipelines_dict=my_pipelines,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    optimize_for='rmse'
)

Training and evaluating 3 models...

--------------------------------------------------
📋 EVALUATION RESULTS
--------------------------------------------------
                    RMSE       MAE      R2
Model                                     
ElasticNet     1252.3995  985.6445  0.5983
Random Forest   432.6809  249.1460  0.9521
XGBoost         417.0433  249.7148  0.9555
--------------------------------------------------
🏆 BEST MODEL (RMSE): XGBoost with score: 417.0433


c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:21:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:21:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


    - Xgboost is best model find in comparision
    - We are hyperparameter tuning Xgboost

In [16]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the parameter grid
# NOTICE the 'model__' prefix. This tells the pipeline to pass these to XGBRegressor.
param_distributions = {
    'model__n_estimators': [100, 300, 500, 700],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [3, 5, 7, 9],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],        # Prevents overfitting by sampling rows
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0]  # Prevents overfitting by sampling columns
}

# 2. Re-instantiate the pipeline (ensuring GPU is enabled for speed)
# We use the same preprocessor built previously
xgb_tune_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(tree_method='hist', device='cuda', random_state=42))
])

# 3. Set up the Randomized Search
random_search = RandomizedSearchCV(
    estimator=xgb_tune_pipeline,
    param_distributions=param_distributions,
    n_iter=20,          # How many random combinations to try (adjust based on your patience)
    cv=3,               # 3-fold cross-validation
    scoring='neg_root_mean_squared_error', # Scikit-learn requires metrics to be maximized, so it uses negative RMSE
    verbose=2,          # Prints progress updates
    random_state=42,
    n_jobs=1            # Leave at 1 when using GPU in Colab to avoid memory conflicts
)

# 4. Run the search
print("Starting Hyperparameter Tuning...")
random_search.fit(X_train, y_train)

# 5. Extract the results
print("\n--- Tuning Complete ---")
print(f"Best Hyperparameters: {random_search.best_params_}")

# The random_search object automatically retrains the best model on the full X_train.
# You can immediately use it to predict:
best_tuned_pipeline = random_search.best_estimator_
tuned_predictions = best_tuned_pipeline.predict(X_test)
print(f"\nTuned RMSE: {np.sqrt(mean_squared_error(y_test, tuned_predictions)):.4f}")
print(f"Tuned R2: {r2_score(y_test, tuned_predictions):.4f}")

Starting Hyperparameter Tuning...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:21:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:21:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=0.9, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=700, model__subsample=0.8; total time=   1.0s


c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:21:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:21:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=0.9, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=700, model__subsample=0.8; total time=   0.9s


...

[CV] END model__colsample_bytree=0.9, model__learning_rate=0.2, model__max_depth=7, model__n_estimators=700, model__subsample=0.8; total time=   2.0s


c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:22:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:22:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=0.9, model__learning_rate=0.2, model__max_depth=7, model__n_estimators=700, model__subsample=0.8; total time=   2.0s


c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:22:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\aksha\Documents\github\traffic-flow-analysis\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:22:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)



--- Tuning Complete ---
Best Hyperparameters: {'model__subsample': 0.8, 'model__n_estimators': 700, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.9}

Tuned RMSE: 401.8647
Tuned R2: 0.9586


  - Saving the pipeline (model with one hot encoder and standard scaler)

In [17]:
model_filename = 'xgboost_traffic_pipeline_predictor.joblib'

joblib.dump(best_tuned_pipeline, model_filename)

print(f"✅ Full pipeline successfully saved to {model_filename}")

✅ Full pipeline successfully saved to xgboost_traffic_pipeline_predictor.joblib
